# Question - Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

### Task 1: Prior Belief Boundaries & Analytical Mean

#### 1. Analytical Expectation of the Prior $\mathbb{E}\left[\Theta^{(0)}\right]$
For a random variable following a Beta distribution $\Theta \sim \text{Beta}(\alpha, \beta)$[cite: 1], the expected value is given analytically by the formula:

$$\mathbb{E}[\Theta] = \frac{\alpha}{\alpha + \beta}$$

Given the initial prior parameters $\alpha = 8$ and $\beta = 1.5$[cite: 1]:

$$\mathbb{E}\left[\Theta^{(0)}\right] = \frac{8}{8 + 1.5} = \frac{8}{9.5} = \frac{16}{19} \approx 0.8421 \text{ (or } 84.21\% \text{ efficiency)}$$


#### 2. Engineering Justification for $\text{Beta}(8, 1.5)$ Prior
This specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy due to the following physical properties:

* **Physical Domain Bounds $\theta \in (0, 1]$:** The Beta distribution is naturally bounded on $[0, 1]$, making it ideal for modeling an efficiency factor where values $> 1.0$ (over-stiffness) or $< 0.0$ (negative stiffness) are physically impossible[cite: 1].
* **Concentration of Mass Near Pristine State ($\theta \to 1.0$):** Because $\alpha = 8 \gg \beta = 1.5$, the probability density function is heavily left-skewed, concentrating the vast majority of its probability mass near $1.0$[cite: 1]. This reflects the realistic engineering baseline that a newly deployed aircraft wing or bridge girder has passed strict quality inspections and is overwhelmingly likely to be undamaged[cite: 1].
* **Soft Lower-Tail Penalty:** The prior assigns near-zero probability density to degraded states ($\theta < 0.5$) before inspection begins[cite: 1], preventing random early sensor noise from triggering false-alarm critical damage reports without substantial accumulating evidence[cite: 1].


#### 3. Python Code to Plot the Prior PDF using Plotly

Below is the Python code to visualize the prior density function over the restricted domain $\theta \in [0.01, 1.0]$[cite: 1]:



In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Define domain theta in [0.01, 1.0]
theta = np.linspace(0.01, 1.0, 1000)

# Beta(8, 1.5) parameters
alpha_p, beta_p = 8.0, 1.5
prior_pdf = beta.pdf(theta, alpha_p, beta_p)
expected_mean = alpha_p / (alpha_p + beta_p)

# Create Plotly figure
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=theta,
    y=prior_pdf,
    mode='lines',
    name='Initial Prior: Beta(8, 1.5)',
    line=dict(color='#2CA02C', width=3)
))

# Expected value vertical line
fig.add_vline(
    x=expected_mean,
    line_width=2,
    line_dash="dash",
    line_color="black",
    annotation_text=f"E[Θ⁽⁰⁾] = {expected_mean:.4f}",
    annotation_position="top left"
)

fig.update_layout(
    title="<b>Task 1: Initial Prior Stiffness Efficiency Density f_Θ⁽⁰⁾(θ) ~ Beta(8, 1.5)</b>",
    xaxis_title="<b>Stiffness Efficiency Factor (θ)</b>",
    yaxis_title="<b>Probability Density</b>",
    template="plotly_white",
    xaxis=dict(range=[0.01, 1.0]),
    hovermode="x unified"
)

fig.show()

---
### Task 2: Structural Likelihood Formulation

#### 1. Derivation of the Single-Step Likelihood Contribution $L(y_k \mid \theta)$

The physical sensor measurement model at inspection step $k$ is given by:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \quad \text{where } \epsilon_k \sim \mathcal{N}(0, \sigma^2)$$

To derive the likelihood probability density function $f_{Y_k \mid \Theta}(y_k \mid \theta)$, we apply the **change of variables** technique for random variables.

##### Step A: Isolate the Noise Random Variable $\epsilon_k$
Taking the natural logarithm of both sides of the physical measurement equation:

$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k \implies \epsilon_k = \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})$$

Notice that $y_k$ is a strictly positive continuous random variable ($y_k > 0$). Conditional on $\Theta = \theta$, the transformed variable $\ln(Y_k)$ follows a Gaussian (Normal) distribution:

$$\ln(Y_k) \mid \Theta = \theta \sim \mathcal{N}\left(\ln(\theta \cdot K_{\text{nominal}}), \sigma^2\right)$$

Therefore, $Y_k \mid \Theta = \theta$ follows a **Log-Normal distribution** with location parameter $\mu_k = \ln(\theta \cdot K_{\text{nominal}})$ and scale parameter $\sigma$.

##### Step B: Change of Variables Probability Density Mapping
Let $g(y_k) = \ln(y_k) = \epsilon_k + \ln(\theta \cdot K_{\text{nominal}})$. By the change of variables theorem:

$$f_{Y_k \mid \Theta}(y_k \mid \theta) = f_{\epsilon_k}\left(g(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right) \cdot \left| \frac{d}{dy_k} g(y_k) \right|$$

Where:
* $f_{\epsilon_k}(\epsilon) = \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{\epsilon^2}{2\sigma^2}\right)$ is the Gaussian PDF of the noise.
* $\frac{d}{dy_k} \ln(y_k) = \frac{1}{y_k}$ is the Jacobian determinant of the transformation.

Substituting these components yields the single-step likelihood contribution $L(y_k \mid \theta)$:

$$L(y_k \mid \theta) = \frac{1}{y_k \cdot \sigma \sqrt{2\pi}} \exp\left( -\frac{\left( \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right)^2}{2\sigma^2} \right), \quad \text{for } y_k > 0$$


#### 2. Joint Likelihood Function for the Running History Vector $y^{(k)}$

Let $y^{(k)} = (y_1, y_2, \dots, y_k)$ represent the vector of continuous sensor readings accumulated up to inspection milestone $k$.

Assuming that the sensor noise values $\epsilon_1, \epsilon_2, \dots, \epsilon_k$ are conditionally independent across inspection time steps given the structural stiffness parameter $\theta$, the joint likelihood function $L\left(y^{(k)} \mid \theta\right)$ is the product of the individual single-step log-normal likelihood contributions:

$$L\left(y^{(k)} \mid \theta\right) = \prod_{i=1}^k L(y_i \mid \theta) = \prod_{i=1}^k \frac{1}{y_i \cdot \sigma \sqrt{2\pi}} \exp\left( -\frac{\left( \ln(y_i) - \ln(\theta \cdot K_{\text{nominal}}) \right)^2}{2\sigma^2} \right)$$

Combining the exponential terms using standard exponent rules gives the compact joint likelihood formulation:

$$L\left(y^{(k)} \mid \theta\right) = \left( \frac{1}{\sigma \sqrt{2\pi}} \right)^k \left( \prod_{i=1}^k \frac{1}{y_i} \right) \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^k \left[ \ln(y_i) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2 \right)$$

---
### Task 3: Mathematical Formulation of the Non-Conjugate Grid Update

#### 1. Explanation of Non-Conjugacy (Absence of Closed-Form Solution)

In Bayesian statistics, a prior distribution is considered **conjugate** to a likelihood function if the resulting posterior distribution belongs to the exact same parametric family of probability distributions as the prior state.

An exact closed-form analytical solution for the posterior density $f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right)$ **does not exist** in this Structural Health Monitoring setup due to the following structural mathematical mismatch:

1. **Prior Distribution Kernel:** The initial state is modeled using a Beta prior, $\Theta \sim \text{Beta}(\alpha_0, \beta_0)$, whose functional form is a polynomial kernel over the domain $\theta \in (0, 1]$:
   $$f_{\Theta}^{(0)}(\theta) \propto \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1}$$

2. **Likelihood Function Kernel:** The physical measurement model yields a Log-Normal likelihood contribution, where the latent variable $\theta$ appears inside a non-linear quadratic logarithmic term within the exponent:
   $$L(y_k \mid \theta) \propto \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right)$$

When multiplying the Beta polynomial kernel $\theta^{\alpha - 1}(1 - \theta)^{\beta - 1}$ by the Log-Normal exponential kernel $\exp\left(-\frac{1}{2\sigma^2}[\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})]^2\right)$, the resulting product cannot be simplified into any standard, named probability density function (such as a Beta, Gaussian, or Gamma family).

Furthermore, the normalizing constant (marginal likelihood integral in the denominator) involves integrating this non-standard transcendental function over the interval $(0, 1]$, which lacks a closed-form antiderivative:

$$Z_k = \int_{0}^{1} f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot L(y_k \mid \theta) \, d\theta$$

Because $Z_k$ cannot be evaluated symbolically, we must rely on numerical grid discretization and quadrature integration.


#### 2. Recursive Posterior Relationship

By Bayes' Theorem, the sequential updating process builds directly on the previous state. The posterior density calculated at inspection step $k-1$ serves directly as the prior distribution for inspection step $k$.

Up to a proportionality constant (omitting the marginal likelihood normalizing constant $Z_k$), the recursive relationship for the posterior density at inspection step $k$ is expressed as:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot L(y_k \mid \theta)$$

Substituting the log-normal single-step likelihood formulation derived in Task 2:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot \frac{1}{y_k \cdot \sigma \sqrt{2\pi}} \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right)$$

For bounded numerical updates, because $y_k$, $\sigma$, and $\sqrt{2\pi}$ are constant relative to $\theta$ at step $k$, the kernel updating equation simplifies compactly to:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right), \quad \theta \in (0, 1]$$

---
### Task 4: Running Point Estimates via Definite Integrals

Because the posterior density $f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right)$ cannot be obtained analytically in closed form, point estimates at inspection milestone $k$ must be evaluated over the bounded physical domain $\theta \in (0, 1]$.

Let $f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) = \frac{\tilde{f}^{(k)}(\theta)}{Z_k}$ denote the fully normalized posterior density at step $k$, where:
* $\tilde{f}^{(k)}(\theta) = f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right)$ is the unnormalized posterior density kernel.
* $Z_k = \int_{0}^{1} \tilde{f}^{(k)}(\theta) \, d\theta$ is the marginal likelihood normalizing constant.


#### 1. The Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$)

The Expected A Posteriori (EAP) estimate represents the expected value (first moment) of the latent structural stiffness parameter $\Theta$ given the accumulated measurement history $y^{(k)}$:

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}\left[\Theta \mid Y^{(k)} = y^{(k)}\right] = \int_{0}^{1} \theta \cdot f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) d\theta$$

Expressed explicitly in terms of the unnormalized posterior kernel $\tilde{f}^{(k)}(\theta)$ and its normalizing integral:

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\int_{0}^{1} \theta \cdot \tilde{f}^{(k)}(\theta) \, d\theta}{\int_{0}^{1} \tilde{f}^{(k)}(\theta) \, d\theta} = \frac{\int_{0}^{1} \theta \cdot f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right) d\theta}{\int_{0}^{1} f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right) d\theta}$$

In a discrete numerical implementation over an $M$-point grid $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$, this ratio of definite integrals is evaluated using composite numerical quadrature (e.g., Simpson's rule or Trapezoidal integration):

$$\hat{\theta}_{\text{Bayes}}^{(k)} \approx \frac{\sum_{m=1}^{M} \theta_m \cdot \tilde{f}^{(k)}(\theta_m)}{\sum_{m=1}^{M} \tilde{f}^{(k)}(\theta_m)}$$


#### 2. The Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$)

The Maximum A Posteriori (MAP) estimate represents the mode—the specific stiffness efficiency value within the physical domain $(0, 1]$ that maximizes the posterior density:

$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right)$$

Since the normalizing constant $Z_k > 0$ is strictly independent of $\theta$, finding the mode of the normalized density is equivalent to finding the location of the peak of the unnormalized density kernel $\tilde{f}^{(k)}(\theta)$:

$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} \left\{ f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right) \right\}$$

In a discrete numerical grid framework, this optimization equation simplifies to finding the grid coordinate index that yields the maximum value:

$$\hat{\theta}_{\text{MAP}}^{(k)} \approx \theta_{m^*}, \quad \text{where } m^* = \arg\max_{m \in \{1, 2, \dots, M\}} \tilde{f}^{(k)}(\theta_m)$$

---
### Task 5: Algorithmic Grid Approximation and Sequential Normalization

Because the log-normal measurement likelihood is non-conjugate with the Beta prior, we maintain and update the posterior density numerically across a discrete domain grid.

#### Numerical Procedure

##### Step 1: Grid Discretization & Boundary Limit Handling
To respect the physical domain constraints $\theta \in (0, 1]$ while avoiding computational division-by-zero or logarithmic singularities ($\ln(0)$) at $\theta = 0$:
1. Define a fine-grained, uniformly spaced grid of $M$ points (e.g., $M = 1000$) over a slightly offset, closed numerical interval $[\theta_{\min}, \theta_{\max}] = [0.001, 1.0]$.
2. Store the grid vector as $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$.
3. Calculate the constant grid spacing step:

$$\Delta \theta = \frac{\theta_{\max} - \theta_{\min}}{M - 1} = \frac{1.0 - 0.001}{M - 1}$$

##### Step 2: Base Prior Initialization ($k = 0$)
1. Evaluate the initial Beta prior density $\text{Beta}(8, 1.5)$ at every grid point $\theta_m$:

$$f_m^{(0)} = \frac{\theta_m^{8 - 1} (1 - \theta_m)^{1.5 - 1}}{B(8, 1.5)}, \quad \text{for } m = 1, 2, \dots, M$$

2. Normalize this initial state vector using the Trapezoidal Rule so that the discrete area sums to $1.0$:

$$\text{Area}^{(0)} = \frac{\Delta \theta}{2} \left[ f_1^{(0)} + 2 \sum_{m=2}^{M-1} f_m^{(0)} + f_M^{(0)} \right], \quad \mathbf{f}^{(0)} = \frac{\mathbf{f}^{(0)}}{\text{Area}^{(0)}}$$

##### Step 3: Sequential Likelihood Evaluation ($k = 1, 2, \dots, n$)
Upon observing a new noisy sensor reading $y_k$:
1. Compute the log-normal likelihood value across all grid points $\theta_m$:

$$L(y_k \mid \theta_m) = \frac{1}{y_k \cdot \sigma \sqrt{2\pi}} \exp\left( -\frac{\left[ \ln(y_k) - \ln(\theta_m \cdot K_{\text{nominal}}) \right]^2}{2\sigma^2} \right), \quad \text{for } m = 1, 2, \dots, M$$

2. Multiply the point-wise likelihood array by the previous step's normalized posterior vector $\mathbf{f}^{(k-1)}$ to obtain the unnormalized posterior array $\tilde{\mathbf{f}}^{(k)}$:

$$\tilde{f}_m^{(k)} = f_m^{(k-1)} \cdot L(y_k \mid \theta_m), \quad \text{for } m = 1, 2, \dots, M$$

##### Step 4: Sequential Normalization via Trapezoidal Rule
1. Calculate the marginal likelihood normalizing constant $Z_k$ by performing numerical quadrature over the unnormalized grid array $\tilde{\mathbf{f}}^{(k)}$ using the Composite Trapezoidal Rule:

$$Z_k = \int_{\theta_{\min}}^{\theta_{\max}} \tilde{f}^{(k)}(\theta) \, d\theta \approx \frac{\Delta \theta}{2} \left[ \tilde{f}_1^{(k)} + 2 \sum_{m=2}^{M-1} \tilde{f}_m^{(k)} + \tilde{f}_M^{(k)} \right]$$

2. Divide every element in the unnormalized vector $\tilde{\mathbf{f}}^{(k)}$ by $Z_k$ to produce the fully normalized posterior state vector for step $k$:

$$f_m^{(k)} = \frac{\tilde{f}_m^{(k)}}{Z_k}, \quad \text{for } m = 1, 2, \dots, M$$

This vector $\mathbf{f}^{(k)}$ becomes the input prior state vector for the next inspection milestone $k + 1$.

### Task 6: Sequential Estimator Tracking and Posterior Density Timeline

#### Analysis of Degradation Convergence and Safety Thresholds

* **Overcoming the Optimistic Prior ($k \approx 3$ to $5$ Steps):**
  At inspection step $k = 0$, the platform assumes an undamaged structure with $\mathbb{E}\left[\Theta^{(0)}\right] \approx 0.842$. Upon receiving noisy physical measurements centered around $\theta_{\text{true}} = 0.68$, the log-normal likelihood exerts a strong corrective pull. Within **3 to 5 continuous inspection readings**, the Bayesian updating mechanism effectively overrides the optimistic initial $\text{Beta}(8, 1.5)$ prior, shifting the probability peak directly over the damaged region ($\theta \approx 0.68$).

* **Narrowing Distribution & Structural Safety Thresholds:**
  As $k$ progresses from step $1$ to $15$, the variance of the posterior density shrinks dramatically (evident in the progressively higher, narrower density peaks in Plot 1).
  * In early steps ($k = 1, 2$), the wide posterior reflects statistical uncertainty, meaning the system cannot rule out a healthy state ($> 0.80$).
  * By step $k = 15$, the density is tightly concentrated around $\theta \in [0.65, 0.71]$. This sharp reduction in variance allows automated structural safety systems to definitively confirm that the component has crossed critical safety thresholds (e.g., a $30\%$ stiffness loss limit) with negligible probability of false alarms.

In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta, lognorm

def simulate_structural_health_monitoring(
    theta_true=0.68,
    K_nominal=50.0,
    sigma=0.15,
    n_steps=15,
    grid_points=1000,
    seed=42
):
    """
    Task 6: Simulates log-normal sensor stream y_k, updates grid posterior density sequentially,
    tracks running EAP and MAP estimates, and produces two interactive Plotly visualizations.
    """
    np.random.seed(seed)

    # 1. Define bounded numerical grid theta in [0.001, 1.0]
    theta_grid = np.linspace(0.001, 1.0, grid_points)
    delta_theta = theta_grid[1] - theta_grid[0]

    # Initialize Beta(8, 1.5) prior
    prior = beta.pdf(theta_grid, 8.0, 1.5)
    running_posterior = prior / np.trapezoid(prior, theta_grid)

    # Store history for point estimates (Step 0 to n_steps)
    bayes_history = [np.trapezoid(theta_grid * running_posterior, theta_grid)]
    map_history = [theta_grid[np.argmax(running_posterior)]]

    # Milestones for full density plot
    milestones = {0, 1, 2, 5, 10, 15}
    density_milestones = {0: running_posterior.copy()}

    # Color mapping for milestones
    colors = {
        0: '#8C8C8C',   # Gray
        1: '#E45756',   # Red-orange
        2: '#2CA02C',   # Green
        5: '#9467BD',   # Purple
        10: '#FF7F0E',  # Orange
        15: '#17BECF'   # Cyan
    }

    # 2. Sequential Simulation
    for k in range(1, n_steps + 1):
        # Draw noisy log-normal sensor reading: y_k = theta_true * K_nominal * exp(N(0, sigma^2))
        epsilon_k = np.random.normal(0.0, sigma)
        y_k = theta_true * K_nominal * np.exp(epsilon_k)

        # Calculate Log-Normal Likelihood L(y_k | theta) over grid
        # Log-normal PDF: 1 / (y_k * sigma * sqrt(2*pi)) * exp(- (ln(y_k) - ln(theta * K_nominal))^2 / (2 * sigma^2))
        mu_grid = np.log(theta_grid * K_nominal)
        log_likelihood = -0.5 * ((np.log(y_k) - mu_grid) / sigma)**2
        likelihood_k = np.exp(log_likelihood) / (y_k * sigma * np.sqrt(2 * np.pi))

        # Unnormalized posterior update
        unnormalized_posterior = running_posterior * likelihood_k

        # Normalize via np.trapezoid
        norm_constant = np.trapezoid(unnormalized_posterior, theta_grid)
        running_posterior = unnormalized_posterior / norm_constant

        # Compute EAP and MAP
        bayes_est = np.trapezoid(theta_grid * running_posterior, theta_grid)
        map_est = theta_grid[np.argmax(running_posterior)]

        bayes_history.append(bayes_est)
        map_history.append(map_est)

        if k in milestones:
            density_milestones[k] = running_posterior.copy()

    # 3. Visualization 1: Progression of Posterior Density Curves
    fig_density = go.Figure()

    for k in sorted(list(milestones)):
        dash_style = 'dash' if k == 0 else 'solid'
        name_str = f"Step {k}: Initial Prior" if k == 0 else f"Step {k}: Density"

        fig_density.add_trace(go.Scatter(
            x=theta_grid,
            y=density_milestones[k],
            mode='lines',
            name=name_str,
            line=dict(color=colors[k], dash=dash_style, width=2.5)
        ))

    fig_density.add_vline(
        x=theta_true,
        line_width=2,
        line_dash="dot",
        line_color="red",
        annotation_text=f"True Stiffness (θ_true = {theta_true})",
        annotation_position="top left"
    )

    fig_density.update_layout(
        title="<b>Structural Health Monitoring: Posterior Density Evolution across Milestones</b>",
        xaxis_title="<b>Stiffness Efficiency Factor (θ)</b>",
        yaxis_title="<b>Probability Density f(θ | y⁽ᵏ⁾)</b>",
        template="plotly_white",
        legend=dict(x=0.02, y=0.98),
        hovermode="x unified",
        xaxis=dict(range=[0.0, 1.0])
    )

    fig_density.show()

    # 4. Visualization 2: Point Estimators Convergence Line Chart
    steps = list(range(n_steps + 1))
    fig_convergence = go.Figure()

    fig_convergence.add_trace(go.Scatter(
        x=steps, y=bayes_history,
        mode='lines+markers',
        name='Posterior Mean (θ_Bayes)',
        line=dict(color='#1F77B4', width=2.5),
        marker=dict(size=6)
    ))

    fig_convergence.add_trace(go.Scatter(
        x=steps, y=map_history,
        mode='lines+markers',
        name='MAP Estimate (θ_MAP)',
        line=dict(color='#D62728', width=2, dash='dot'),
        marker=dict(size=6)
    ))

    fig_convergence.add_hline(
        y=theta_true,
        line_width=2,
        line_dash="dash",
        line_color="black",
        annotation_text=f"True Efficiency (θ_true = {theta_true})",
        annotation_position="bottom right"
    )

    fig_convergence.update_layout(
        title="<b>Sequential Convergence of Structural Estimators (θ_true = 0.68)</b>",
        xaxis_title="<b>Inspection Time Step (k)</b>",
        yaxis_title="<b>Estimated Stiffness Factor (θ)</b>",
        template="plotly_white",
        legend=dict(x=0.65, y=0.98),
        hovermode="x unified",
        xaxis=dict(dtick=1),
        yaxis=dict(range=[0.5, 1.0])
    )

    fig_convergence.show()

# Run simulation
simulate_structural_health_monitoring()